In [1]:
#LASSO Logistic Regression with Symptom Interactions
# Import required libraries
# This section loads Python libraries needed for data manipulation,
# machine learning modeling, and likelihood calculations.

import pandas as pd
import numpy as np

from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss



In [2]:
# SECTION 2: Load the dataset
# The dataset contains COVID test results and reported symptoms.

data = pd.read_excel("~/Downloads/D483.xlsx")



# SECTION 3: Create binary outcome variable
# Convert the COVID test result to a binary outcome where
# 1 indicates a positive COVID diagnosis and 0 indicates negative.

data["covid_positive"] = data["TestResult"].str.contains("positive", case=False).astype(int)

In [18]:
# This code applies LASSO logistic regression to identify which symptoms and combinations
# of symptoms are most predictive of a COVID-19 diagnosis. Polynomial feature generation
# expands the original symptom variables to include individual symptoms, symptom pairs,
# and symptom triplets, allowing the model to capture clinically meaningful symptom
# patterns rather than relying only on single predictors.

# LASSO regularization performs automatic feature selection by shrinking weak predictors
# toward zero while retaining the most informative symptoms and symptom combinations.
# The model is estimated using three regularization levels (C = 0.1, 0.01, and 0.001)
# to examine the stability of predictors under different levels of shrinkage.

# Model performance is evaluated using McFadden’s pseudo R², which compares the fitted
# model with a baseline model containing no predictors. This metric measures how much
# symptom information improves prediction of COVID diagnosis. In practice, McFadden R²
# values typically range from 0 (worst fit) to about 0.2–0.4 for strong logistic models.
# The results are also separated into individual symptoms, symptom pairs, and symptom
# triplets, helping identify the most important symptom clusters associated with COVID.

# SECTION 4: Select symptom predictor variables
# These symptoms will be used to predict the probability of COVID infection.

symptom_columns = [
    "symptomsresp",
    "symptomsconsresp",
    "symptomsgastro",
    "symptomsconsgastro",
    "symptomsneuro",
    "symptomsconsneuro",
    "symptomsinflamm",
    "symptomsconsinflamm",
    "symptomsfirst"
]


df = data[symptom_columns + ["TestResult"]].dropna()

X = df[symptom_columns]
print(X.head(2))

y = df["TestResult"]

# convert outcome variable to binary if needed
if y.dtype == "object":
    y = y.str.contains("positive", case=False).astype(int)

                             symptomsresp symptomsconsresp  \
28   Cough,Shortness of breath,Chest pain               No   
51  Runny nose,Cough,Wheezing,Sore throat               No   

                            symptomsgastro symptomsconsgastro  \
28           Change in or loss of appetite                 No   
51  Diarrhea,Change in or loss of appetite                 No   

                            symptomsneuro symptomsconsneuro  \
28            Loss of smell,Loss of taste                No   
51  Headaches,Loss of smell,Loss of taste                No   

                                      symptomsinflamm symptomsconsinflamm  \
28  Unexplained rashes anywhere else,Excessive swe...                  No   
51                   Unexplained rashes anywhere else                  No   

                                        symptomsfirst  
28  Chills,Excessive sweating,Fatigue (more than n...  
51                              Headaches,Sore throat  


In [19]:
# Generate symptom interaction terms
# PolynomialFeatures creates interaction predictors including
# individual symptoms, symptom pairs, and symptom triplets.

# check that symptom variables exist
if len(symptom_columns) == 0:
    raise ValueError("No symptom columns found in the dataset.")

# check that predictor matrix is not empty
if X.shape[0] == 0:
    raise ValueError("Dataset is empty after preprocessing.")

# ensure data are numeric
X = X.apply(pd.to_numeric, errors="coerce").fillna(0)

poly = PolynomialFeatures(
    degree=3,
    interaction_only=True,
    include_bias=False
)

X_poly = poly.fit_transform(X)

feature_names = poly.get_feature_names_out(symptom_columns)


# Show how many predictors were generated
# This illustrates the feature expansion created by interaction modeling.

print("\nNumber of original symptoms:", len(symptom_columns))
print("Total predictors after interactions:", len(feature_names))


Number of original symptoms: 9
Total predictors after interactions: 129


In [20]:
# Compute null model likelihood
# This is needed to calculate McFadden pseudo R-square.

p_null = np.mean(y)

null_log_likelihood = np.sum(
    y * np.log(p_null) + (1 - y) * np.log(1 - p_null)
)


In [25]:
# Function to run LASSO logistic regression
# This fits the model and reports non-zero coefficients.

def run_lasso(C_value):

    model = LogisticRegression(
        penalty="l1",
        solver="liblinear",
        C=C_value,
        max_iter=10000
    )

    model.fit(X_poly, y)

    probabilities = model.predict_proba(X_poly)[:,1]

    model_log_likelihood = -log_loss(y, probabilities, normalize=False)

    pseudo_r2 = 1 - (model_log_likelihood / null_log_likelihood)

    coefficients = model.coef_[0]

    results = pd.DataFrame({
        "feature": feature_names,
        "coefficient": coefficients
    })

    non_zero = results[results["coefficient"] != 0]

    print("\nLASSO Results for C =", C_value)
    print("McFadden Pseudo R-square =", round(pseudo_r2,4))
    print("Number of selected predictors =", len(non_zero))

    return non_zero, pseudo_r2


In [31]:
# Run LASSO models with three hyperparameters

results_C1, r2_C1 = run_lasso(1)
results_C2, r2_C2 = run_lasso(0.1)
results_C3, r2_C3 = run_lasso(0.01)



# Combine results from all models

all_results = pd.concat([
    results_C1.assign(C=1),          #0.1, 0.01, 0.001 penatly will be too small
    results_C2.assign(C=0.1),
    results_C3.assign(C=0.01)
])

all_results["absolute_weight"] = abs(all_results["coefficient"])




LASSO Results for C = 1
McFadden Pseudo R-square = -0.0003
Number of selected predictors = 0

LASSO Results for C = 0.1
McFadden Pseudo R-square = -0.0265
Number of selected predictors = 0

LASSO Results for C = 0.01
McFadden Pseudo R-square = -0.0265
Number of selected predictors = 0


In [33]:
# Separate symptom effects into single symptoms, pairs, and triplets

all_results["symptom_count"] = all_results["feature"].str.count(" ") + 1

single_symptoms = all_results[all_results["symptom_count"] == 1]
symptom_pairs = all_results[all_results["symptom_count"] == 2]
symptom_triplets = all_results[all_results["symptom_count"] == 3]


# Rank the most important predictors by absolute coefficient value

top_single = single_symptoms.sort_values(
    by="absolute_weight",
    ascending=False
).head(10)

top_pairs = symptom_pairs.sort_values(
    by="absolute_weight",
    ascending=False
).head(10)

top_triplets = symptom_triplets.sort_values(
    by="absolute_weight",
    ascending=False
).head(10)


# Display results

print("\nTop Individual Symptoms")
print(top_single[["feature","coefficient","C"]])

print("\nTop Symptom Pairs")
print(top_pairs[["feature","coefficient","C"]])

print("\nTop Symptom Triplets")
print(top_triplets[["feature","coefficient","C"]])


Top Individual Symptoms
Empty DataFrame
Columns: [feature, coefficient, C]
Index: []

Top Symptom Pairs
Empty DataFrame
Columns: [feature, coefficient, C]
Index: []

Top Symptom Triplets
Empty DataFrame
Columns: [feature, coefficient, C]
Index: []
